# 10.8 Astropy for Scientific Data Analysis

Astropy is a Python library for astronomy and scientific data. It helps us work safely with **physical units**, **dates and times**, **sky coordinates**, and **tables that remember units**.

## What you will learn

1. Create values that carry units.
2. Convert units without doing conversion arithmetic by hand.
3. Calculate with astronomical times.
4. Store and compare positions in the sky.
5. Filter an Astropy table and create a derived column.
6. Convert a scientific table to Pandas when another tool needs it.

## Installation

Run the next command once if Astropy is missing. `%pip` installs into the same Python environment as this notebook. Remove the `#` and run it only when needed.

In [ ]:
# %pip install astropy

In [ ]:
# NumPy supplies mathematical functions for derived columns.
import numpy as np
# Astropy itself lets us display the installed version.
import astropy
# SkyCoord represents a position in the sky.
from astropy.coordinates import SkyCoord
# QTable is a table whose columns can keep physical units.
from astropy.table import QTable
# Time understands astronomical dates and time scales.
from astropy.time import Time
# The short name u makes unit expressions easy to read.
import astropy.units as u

# Printing versions makes a notebook easier to reproduce later.
print("Astropy version:", astropy.__version__)

## 1. Quantities and units

A normal number such as `150` does not tell us whether it means metres, kilometres, or seconds. Multiplying by an Astropy unit creates a `Quantity`. Astropy then checks compatible units and performs conversions for us.

In [ ]:
# This distance remembers that its unit is kilometres.
distance = 384_400 * u.km
# This duration remembers that its unit is hours.
travel_time = 72 * u.hour
# Dividing distance by time automatically creates a speed quantity.
average_speed = distance / travel_time
# The to method converts the result to compatible requested units.
speed_km_per_second = average_speed.to(u.km / u.s)

# Display the original value and two safe conversions.
print("Earth–Moon distance:", distance)
print("Distance in metres:", distance.to(u.m))
print("Average speed:", speed_km_per_second.round(2))

### Why units matter

Units prevent silent mistakes. Astropy allows metres plus kilometres because they describe the same physical type, but it rejects metres plus seconds because distance and time cannot be added. Use `.to(...)` for a new converted quantity and `.value` only when a library explicitly requires plain numbers.

## 2. Astronomical time

`Time` can represent dates in formats such as ISO strings, Julian Date, and Modified Julian Date. Subtracting two `Time` objects returns a `TimeDelta`.

In [ ]:
# Create the launch instant for the James Webb Space Telescope.
launch = Time("2021-12-25 12:20:00", scale="utc")
# Create another UTC instant for this small example.
observation = Time("2022-07-12 14:30:00", scale="utc")
# Subtraction produces an exact time difference.
elapsed = observation - launch

# ISO is friendly for people, while JD is common in astronomy.
print("Launch in ISO format:", launch.iso)
print("Launch as Julian Date:", launch.jd)
print("Elapsed days:", elapsed.to_value(u.day).round(2))

## 3. Sky coordinates

A sky position needs values, units, and a coordinate frame. Right ascension (`ra`) behaves like celestial longitude; declination (`dec`) behaves like celestial latitude. ICRS is a standard modern frame.

In [ ]:
# Define Sirius using right ascension and declination in degrees.
sirius = SkyCoord(ra=101.287155 * u.deg, dec=-16.716116 * u.deg, frame="icrs")
# Define Betelgeuse in the same frame and units.
betelgeuse = SkyCoord(ra=88.792939 * u.deg, dec=7.407064 * u.deg, frame="icrs")
# Separation calculates the angle between two positions on a sphere.
angular_distance = sirius.separation(betelgeuse)
# Astropy can transform the same position into the Galactic frame.
sirius_galactic = sirius.galactic

# Show the angle and the transformed longitude and latitude.
print("Angular separation:", angular_distance.to(u.deg).round(2))
print("Sirius Galactic longitude:", sirius_galactic.l.round(2))
print("Sirius Galactic latitude:", sirius_galactic.b.round(2))

## 4. Unit-aware tables and analysis

`QTable` resembles a Pandas DataFrame, but scientific columns can preserve units. Below, apparent magnitude describes how bright a star appears from Earth. We calculate absolute magnitude, an estimate of brightness at a standard distance of 10 parsecs.

In [ ]:
# Build a small table of familiar stars.
stars = QTable()
# Text columns work like ordinary table columns.
stars["name"] = ["Sirius", "Betelgeuse", "Vega", "Polaris"]
# This distance column keeps parsecs as part of the data.
stars["distance"] = [2.64, 168.0, 7.68, 137.0] * u.pc
# Magnitude is a logarithmic, dimensionless brightness scale.
stars["apparent_magnitude"] = [-1.46, 0.42, 0.03, 1.98]
# Dividing by one parsec gives a unitless number for logarithm calculations.
distance_values = stars["distance"].to_value(u.pc)
# The distance-modulus formula creates a useful derived feature.
stars["absolute_magnitude"] = stars["apparent_magnitude"] - 5 * np.log10(distance_values / 10)

# Boolean filtering keeps stars closer than ten parsecs.
nearby_stars = stars[stars["distance"] < 10 * u.pc]
# Sorting places the closest star first.
nearby_stars.sort("distance")
# Display both the complete data and filtered result.
print(stars)
print("\nStars closer than 10 parsecs:")
print(nearby_stars)

## 5. Convert to Pandas

Use Pandas when you need a library that expects a DataFrame. Unit metadata does not behave exactly like a `QTable`, so convert unit columns to the units you want first.

In [ ]:
# Copy the table so the original unit-aware data remains unchanged.
export_table = stars.copy()
# Replace the Quantity with explicit numeric parsec values before export.
export_table["distance_pc"] = export_table["distance"].to_value(u.pc)
# Remove the original unit-bearing column from this export copy.
export_table.remove_column("distance")
# Convert the remaining table into a Pandas DataFrame.
stars_pandas = export_table.to_pandas()
# Display the object type and preview.
print(type(stars_pandas))
stars_pandas

## Summary

- `Quantity` keeps a number and physical unit together.
- `Time` safely handles astronomical dates, formats, and differences.
- `SkyCoord` understands coordinate frames, transformations, and angular separation.
- `QTable` keeps units attached to scientific table columns.
- Convert deliberately when moving unit-aware data into Pandas.

## Practice

1. Convert 1 astronomical unit (`u.au`) to kilometres.
2. Calculate the days between two dates important to you.
3. Add another star and filter stars farther than 100 parsecs.
4. Find the angular separation between Vega and Polaris using `SkyCoord`.
5. Explain why removing units too early can cause analysis mistakes.